# Building a Reflection Agent with LangGraph

This project builds a **reflection agent** with LangGraph — a self-improving AI system that evaluates its own output, identifies weaknesses, and refines the result over several iterations. It implements a graph-based workflow that mirrors how a person drafts, critiques, and rewrites: a *generation* node produces content, a *reflection* node critiques it, and a conditional edge loops between them until the result is good enough.

The worked example is a LinkedIn post generator that turns a rough first draft into a polished, engaging post — but the same pattern applies to any task where a first attempt benefits from review, such as writing code, drafting emails, or summarizing documents.

## Table of Contents

1. What is Reflection?
2. Workflow of a Reflection Agent in LangGraph
3. Setup
4. Building the post generator — prompts and chains
5. Agent state and `MessageGraph`
6. Generation and reflection nodes
7. Edges, entry point, and the router
8. Compiling and running the workflow
9. Visualizing the graph

## What this project covers

- Building reflection-enabled agents on LangGraph's graph-based workflow structure
- A multi-step process that generates, evaluates, and refines AI-produced content
- Conversational workflows with message state management for context retention
- Conditional routing logic to control agent behavior and iteration count
- Applying reflection to improve the quality of generated content
- Designing self-improving systems that address their own limitations

----


## Setup


This project uses the following libraries:

*   [`langgraph`](https://langchain-ai.github.io/langgraph/) — build state-based workflows and graphs for agent systems.
*   [`langchain`](https://python.langchain.com/docs/get_started/introduction) — build LLM-powered applications and workflows.
*   [`langchain-groq`](https://pypi.org/project/langchain-groq/) — call Groq-hosted LLMs through LangChain.
*   [`python-dotenv`](https://pypi.org/project/python-dotenv/) — load the `GROQ_API_KEY` from a `.env` file.

### Installing Required Libraries


In [ ]:
%pip install -q langgraph langchain langchain-groq python-dotenv

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.graph import END, MessageGraph, StateGraph

from typing import List, Sequence
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# What is Reflection?
Reflection is a prompting strategy aimed at enhancing the quality and accuracy of outputs generated by AI agents. It involves getting the agent to **pause, review, and critique** its own outputs before finalizing them. This iterative process helps in reducing errors and improving performance over time.

For example, when an AI model generates code, it typically outputs the result instantly. However, just like human programmers, code needs to be tested and refined. Reflection ensures that the AI agent **evaluates the generated code, identifies potential errors, and iterates** to fix them. This mimics how developers write, test, debug, and optimize their work, resulting in more reliable outputs.

A simple analogy is comparing it to having two systems:
- **System 1** – Reactive and instinctive (quick initial responses).
- **System 2** – Reflective and deliberate (carefully reviewing and refining outputs).

Reflection agents encourage AI to function more like **System 2**, iterating over their work until the desired quality is achieved.


## Workflow of Reflection Agent in LangGraph:

1. **Generation Node: Generate Initial Output**
   - The first step in the process is the **generation node**, which quickly produces an initial output based on the given prompt. This stage is all about generating a first draft without focusing too much on perfection. It acts like an instinctive response, providing a rough version of the output. For example, if the task is to write a LinkedIn post, the generation node would come up with a basic idea. This draft is then passed to the next step for evaluation.
     
<br>

2. **Evaluation Node: Evaluate Output for Quality**
   - After the initial output is generated, the **evaluation node** assesses its quality. This step is about checking if the output is good enough or if it needs improvement. The evaluation focuses on key aspects like whether the message is clear, engaging, and relevant. For instance, in the case of a LinkedIn post, the evaluation might decide if the post feels authentic, aligns with professional tone, or misses important context. If the output is deemed acceptable, it moves forward to the final step.

<br>

3. **Reflection Node: Critique and Refine**
   - If the evaluation node determines that the output needs improvement, the **reflection node** steps in to refine the content. This step is more thoughtful and deliberate, where the system reflects on the output and looks for ways to improve it. The reflection node critiques the draft, suggests changes, and revises the content to make it more polished. It could involve enhancing tone, adding clarity, highlighting achievements, or making the post more engaging. The system keeps refining the output through this process until it reaches the desired quality level.
     
<br>

4. **Final Output: Present Refined Result**
   - Once the reflection node has done its job, the final output is produced. This is the result of the reflection and evaluation processes, where the initial draft has been refined and improved. The agent now presents the final version of the content — a polished, high-quality LinkedIn post ready for publishing. After this step, the process concludes, and the final response is delivered to the user.


---

**Example (LinkedIn Post Generation):**  
Imagine asking an AI to write a LinkedIn post announcing a job promotion:  

**Prompt:**  
"Write a LinkedIn post announcing my promotion to Engineering Manager."  

**AI’s Initial Output (System 1):**  
*"Excited to share that I’ve been promoted to Engineering Manager!"*  

**Reflection Step:**  
The AI reviews the post and asks:  
*"Does this post highlight leadership growth or express gratitude?"*  

**Refined Output (System 2):**  
*"I'm thrilled to share that I've been promoted to Engineering Manager at [Company]! Grateful for the mentorship, team collaboration, and opportunities that led to this moment. Looking forward to leading new initiatives and continuing to grow with this incredible team. #Leadership #CareerGrowth #EngineeringManager"*

---


### Building an Optimized LinkedIn Post Generator with a Reflection Agent

The goal is to improve the quality of a generated post by letting the model critique its own output and refine it iteratively based on that feedback, which lifts engagement, relevance, and tone.

The system has a **generation phase** that writes the post and a **reflection phase** that reviews and improves it. Cycling between the two produces a stronger final result than a single pass.

### Instantiating the Language Model

In [ ]:
# Configure the LLM (Groq-hosted Llama). Create a .env file with GROQ_API_KEY=...
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))  # searches up the directory tree for a .env file

from langchain_groq import ChatGroq

# Temperature > 0 gives the generator some creative latitude; the reflector critiques what it writes.
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

### **Generation Prompt for Posts**

In this section, we are creating a generation prompt for generating LinkedIn posts. The assistant is tasked with crafting high-quality post based on the user's input. Additionally, if the user provides feedback or critique, the assistant revises the post content accordingly.


We are using **`ChatPromptTemplate`** from LangChain to structure the prompt. The prompt has two main parts:

1. **System Message**:  
   This provides instructions to the assistant about its role and task.  
   Here, the assistant is framed as a **professional LinkedIn content assistant** who is expected to generate the best possible LinkedIn post based on the user's input.  
   It also specifies that if the user provides feedback or critique, the assistant should revise the post to improve clarity, tone, or engagement.

2. **MessagesPlaceholder**:  
   This is used to inject the actual content or message that the post will be based on.  
   The placeholder will be populated with the user’s request at runtime.


In [ ]:
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a professional LinkedIn content assistant tasked with crafting engaging, insightful, and well-structured LinkedIn posts."
            " Generate the best LinkedIn post possible for the user's request."
            " If the user provides feedback or critique, respond with a refined version of your previous attempts, improving clarity, tone, or engagement as needed.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

### **Creating the Chain for LinkedIn Post Generation**

In this step, we are combining the **`generation_prompt`** with a language model (LLM) to form a complete chain that will allow the system to generate LinkedIn posts based on user input.

The **`generate_chain`** links the **`generation_prompt`** with the **LLM** (Large Language Model), enabling the system to generate a professional LinkedIn post after processing the user's input through the prompt.

- **`generation_prompt`**: This is the template that guides the model on how to generate the LinkedIn post, including the system message and the placeholder for the user's input.
- **`llm`**: This is the language model that will take the prompt and produce the post based on the input provided.

By using the pipe operator (`|`), we are chaining these components together so that the prompt flows seamlessly into the language model and the model generates the final LinkedIn post.


In [ ]:
generate_chain = generation_prompt | llm

<br>

### **Reflection Prompt for LinkedIn Post Critique**

In this step, we define the **`reflection_prompt`**, which is a template used for generating critiques and recommendations to improve a user's LinkedIn post. This prompt guides the model to assess the quality of a LinkedIn post and provide structured, actionable feedback.

- **`reflection_prompt`**: The system message here instructs the model to act as a professional LinkedIn content strategist. The model will evaluate the post based on factors like tone, structure, clarity, engagement potential, formatting, and relevance. It then generates feedback to help improve the post’s overall effectiveness and alignment with LinkedIn best practices.


In [ ]:
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional LinkedIn content strategist and thought leadership expert. Your task is to critically evaluate the given LinkedIn post and provide a comprehensive critique. Follow these guidelines:

        1. Assess the post’s overall quality, professionalism, and alignment with LinkedIn best practices.
        2. Evaluate the structure, tone, clarity, and readability of the post.
        3. Analyze the post’s potential for engagement (likes, comments, shares) and its effectiveness in building professional credibility.
        4. Consider the post’s relevance to the author’s industry, audience, or current trends.
        5. Examine the use of formatting (e.g., line breaks, bullet points), hashtags, mentions, and media (if any).
        6. Evaluate the effectiveness of any call-to-action or takeaway.

        Provide a detailed critique that includes:
        - A brief explanation of the post’s strengths and weaknesses.
        - Specific areas that could be improved.
        - Actionable suggestions for enhancing clarity, engagement, and professionalism.

        Your critique will be used to improve the post in the next revision step, so ensure your feedback is thoughtful, constructive, and practical.
        """
    ),
    MessagesPlaceholder(variable_name="messages")
])

### **Creating the Reflect Chain**
The **`reflect_chain`** is created by combining the **`reflection_prompt`** with the language model (LLM). This chain allows the model to evaluate and provide feedback on the generated post.


In [ ]:
reflect_chain = reflection_prompt | llm

### **Defining the Agent State for Reflection Agent**

When building a conversational workflow from scratch, the **state** represents the evolving context of the conversation or task. It tracks the interactions between the user and the AI, growing dynamically as new messages are added. 

If we were to define the state manually, it would look like this:

---

#### **Manual State Definition**

Using Python's `TypedDict`, we can define a state that holds a list of messages:

```python
from typing import List, Annotated, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Define State with TypedDict
class AgentState(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage | SystemMessage], "add_messages"]
```

In this setup:  
- **`HumanMessage`**: Represents user inputs or prompts.  
- **`AIMessage`**: Represents AI-generated responses.  
- **`SystemMessage`**: Represents system-level instructions, such as refinement feedback or evaluation criteria.  
- **`add_messages`**: Ensures new messages are appended to the list, preserving the context needed for iterative interactions.

While this approach is flexible, it requires manual management of the state, including creating workflows and maintaining the message list.


---
#### **LangGraph's `MessageGraph`**

Instead of manually defining and managing the state, **LangGraph** offers a **prebuilt solution** called `MessageGraph`. It abstracts the complexity of state management, making it easy to create and work with conversational workflows.

**Features of `MessageGraph`:**  
1. **Predefined State Management**: Handles the underlying state representation for you, similar to what was manually defined above.  
2. **Ease of Integration**: Provides a seamless way to interact with LLMs by managing conversation states automatically.  
3. **Workflow Simplification**: Streamlines the process of building workflows with less boilerplate code.  

---


#### **Initializing `MessageGraph`**

To streamline workflow creation and state management, LangGraph provides a prebuilt solution called `MessageGraph`. This simplifies the process of setting up conversational workflows by handling the underlying structure automatically.


In [ ]:
from langgraph.graph import MessageGraph
from typing import List, Annotated, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Initialize a predefined MessageGraph
graph = MessageGraph()

Behind the Scenes:

- As the user interacts with the agent, **`HumanMessage`** is added to the state.  
- The AI generates a response (**`AIMessage`**), which gets appended to the list.  
- If the output needs improvement, a **`SystemMessage`** can trigger a reflection phase to refine the response.



### **Defining the Generation and Reflection  Node**

The `generation_node` function acts as the starting point in the Reflection Agent's workflow. It generates an initial output based on the current state of the conversation, which contains all previous messages (user inputs, AI responses, and system instructions). 

- **Input**: The function accepts the `state`, which is a sequence of `BaseMessage` objects (i.e., `HumanMessage`, `AIMessage`, `SystemMessage`). These messages provide the context necessary for generating a meaningful response.

- **Output**: The function uses the `generate_chain` to produce an output by invoking the chain with the `state` as input. The `invoke` function triggers the execution of the chain, where the `messages` in the state guide the chain's generation process. The output is generated based on the context provided by these messages, ensuring that the response is appropriate to the current stage of the conversation or task.


In [ ]:
def generation_node(state: Sequence[BaseMessage]) -> List[BaseMessage]:
    generated_post = generate_chain.invoke({"messages": state})
    return [AIMessage(content=generated_post.content)]



The `reflection_node` function plays a key role in improving the output generated in the `generation_node`. It critiques the original output and makes recommendations for refinement. The feedback mechanism helps enhance the final result, making it more in line with the desired outcome, whether that involves clarity, engagement, or tone adjustments.

- **Input**: The function takes `messages`, which is a sequence of `BaseMessage` objects. This includes previous AI responses, user inputs, and system-level instructions. The messages are used to provide context to the reflection process, guiding the generation of a more refined output.
  
- **Output**: The function invokes `reflect_chain`, passing the `messages` as input to critique and improve the content. After receiving the refined output, it returns the result as a `HumanMessage` object.


In [ ]:
def reflection_node(messages: Sequence[BaseMessage]) -> List[BaseMessage]:
    """Critique the latest draft and return the feedback as a HumanMessage.

    The roles are flipped before calling the reflector: the post that the
    generator produced (an AIMessage) is presented to the critic as a
    HumanMessage. Without this, the critic sees a conversation that already
    ends with an assistant turn and often replies with nothing at all.
    """
    cls_map = {"ai": HumanMessage, "human": AIMessage}
    translated = [messages[0]] + [
        cls_map[m.type](content=m.content) for m in messages[1:]
    ]

    res = reflect_chain.invoke({"messages": translated})
    # Returned as a HumanMessage so the generator treats it as feedback to act on.
    return [HumanMessage(content=res.content)]

<br>

### **Why `HumanMessage`?**

The output is wrapped in a `HumanMessage` because the reflection process is a form of feedback or critique given to the **generation agent**, and the feedback is intended to be treated as if it is coming from the user. This is important for the iterative process where the AI generates content and then receives human-like feedback to improve the output. In the context of this workflow, we treat the feedback as if a human is guiding the reflection agent to enhance its output.

- **HumanMessage** here is not used to represent user input directly but rather to provide feedback (as if from a human perspective). This feedback is passed back into the system, enabling the generation agent to revise its content. 
- It effectively gives the reflection node the authority to "speak" to the generation node, but in the context of providing critique and recommendations for refinement.


### **Adding the Generate Node to the Graph**

Now we add the generation node to the graph using the `add_node` function. This function takes two parameters:  

1. **Name**: A unique identifier for the node, in this case, `"generate"`.  
2. **Function**: The function to be executed when this node is triggered, here `generation_node`.  


In [ ]:
graph.add_node("generate", generation_node)

To summarize the process: the generation prompt is chained to the LLM to form the generate chain. The generate node invokes that chain and stores the LLM's output in a sequence of messages, and `add_node` registers it on the graph.

<br>

### **Adding the Reflect Node to the Graph**

We now add the reflection node to the graph using the `add_node` function. This function takes two parameters:  

1. **Name**: A unique identifier for the node, in this case, `"reflect"`.  
2. **Function**: The function to be executed when this node is triggered, here `reflection_node`.  

This step integrates the `"reflect"` node into the graph, linking it to the `reflection_node` function. This node is responsible for providing feedback and suggestions for improving the generated content, enabling the reflection phase of the agent.


In [ ]:
graph.add_node("reflect", reflection_node)

The reflect node works the same way: `reflect_chain.invoke({"messages": messages})` maps its input to the sequence of messages sent to the LLM and returns the critique as a new message in a list. `add_node` adds it to the graph.

`graph.add_edge("reflect", "generate")` creates a one-way connection from the reflect node back to the generate node. It is a direct path telling the workflow "once reflection is done, go back to generation."

In [ ]:
graph.add_edge("reflect", "generate")


<br>

### **Setting the Entry Point in the Graph**

The `graph.set_entry_point(GENERATE)` specifies where the workflow begins in the agent's graph. By setting **`GENERATE`** as the entry point, the process starts with the generation node, which creates the initial response based on the provided state.


In [ ]:
graph.set_entry_point("generate")

<br>

### **Adding a Router Node for Decision Making**

The router node in the graph is responsible for determining whether the workflow should proceed to the reflection phase or terminate. This decision can be made in two ways:  

1. **Predefined Logic**:  
   A simple condition is used to check the number of messages in the state. If the number of messages exceeds 6 (`len(state) > 6`), the workflow ends. Otherwise, it continues to the reflection phase.  

2. **LLM-Based Logic**:  
   Instead of relying on predefined logic, we can integrate an LLM to evaluate the context of the messages and decide whether further reflection is necessary.  

For now, we will implement the predefined logic of checking the message count. Later, we will enhance this functionality by incorporating an LLM to decide dynamically whether to proceed to the reflection phase or end the workflow.


In [ ]:
def should_continue(state: List[BaseMessage]):
    """Route the workflow: keep reflecting until the message history is long enough.

    Each generate/reflect round adds two messages, so a limit of 6 gives roughly
    three generation passes before the workflow ends.
    """
    if len(state) > 6:
        return END
    return "reflect"

Since the LLM must decide whether to continue or end the process, we use the `add_conditional_edges` method to handle two possible paths: if the maximum iterations have not been reached, continue by sending messages from generate to reflect shown by the edge between the green and blue node; if the maximum is met, go to the end node (represented by a square).


In [ ]:
graph.add_conditional_edges("generate", should_continue)

### **Compiling the Workflow**

Now we compile the workflow using `graph.compile()`. This step ensures that all nodes, edges, and conditional logic defined in the graph are connected and ready for execution.  


In [ ]:
workflow = graph.compile()

### **Defining Inputs for the Workflow**

In this example, we define the initial user input as a `HumanMessage`. This message contains a request to improve a post related to LangChain's Tool Calling feature. The content provides the context for the workflow, which will process it and generate a refined output.


In [ ]:
# The task for the agent. Swap this out for any prompt you want refined.
inputs = HumanMessage(
    content="""Write a LinkedIn post about landing your first software developer job, under 160 characters."""
)

### **Executing the Workflow**

Once the input has been defined, the workflow is executed using the `graph.invoke()` method. This processes the `inputs` through the workflow graph, starting from the entry point, and generates a response.


In [ ]:
response = workflow.invoke(inputs)

Let's look at the first generated post, before any critique.

In [ ]:
response[1].content

Now, let's see the first critique for this generated post.


In [ ]:
response[2].content

---
As we can see, this is the first critique of our generated post. It highlights both strengths and areas for improvement, offering specific suggestions to enhance engagement, relevance, and impact. Next, we’ll refine the post based on this feedback and generate an improved version.


Now, let's see the final generated response after multiple iterations incorporating the feedback.


In [ ]:
response[-1].content

This table tracks the state transitions in a reflection agent's workflow. Each row represents a step in the process, showing how the state evolves from the initial user input through multiple iterations of generation and reflection. The table captures the iteration number, message type (Human/AI/System), current state, active node (Input/Generate/Reflect), and where the workflow goes next. After 3 iterations, reaching 6 total state changes, the workflow terminates at END.


| Iteration | Type | State | Node | Next Action |
|-----------|------|-------|------|-------------|
| 1 | Human | Initial request | Input | Generate |
| 1 | AI | Generated content | Generate | Reflect |
| 1 | System | Reflection feedback | Reflect | Generate |
| 2 | AI | Revised content | Generate | Reflect |
| 2 | System | Refinement feedback | Reflect | Generate |
| 3 | AI | Final content | Generate | END |


---
The final post reflects the accumulated feedback — a stronger hook, a clearer call to action, and tighter phrasing — making it more engaging than the first draft.

#### Plotting the Graph

Finally, render the workflow graph as an image to see the generate/reflect loop and the conditional edge that ends the run.

In [ ]:
from IPython.display import Image, display

# Render the compiled graph. draw_mermaid_png() needs no extra system packages
# (it renders remotely); if it is unavailable, print the Mermaid definition instead.
try:
    display(Image(workflow.get_graph().draw_mermaid_png()))
except Exception:
    print(workflow.get_graph().draw_mermaid())

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)